# 02 — Feature Engineering

**Goal**: Create agronomically meaningful features from raw sensor data:
- Growing Degree Days (GDD)
- Temperature stress indicators
- Soil moisture deficit
- Weather aggregates (rolling means)
- Crop-specific interaction terms

**Reference**: FAO-56, Zalom et al. (1983) for degree-day calculation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
%matplotlib inline

print("Libraries loaded.")

## 1. Load Data

In [ ]:
np.random.seed(42)
n_samples = 10000

crop_types = ['banana', 'maize', 'cacao', 'rice']
base_yields = {'banana': 35000, 'maize': 6000, 'cacao': 800, 'rice': 4500}

df = pd.DataFrame({
    'crop_type': np.random.choice(crop_types, n_samples),
    'temp_mean': np.random.normal(25, 5, n_samples),
    'temp_max': np.random.normal(32, 4, n_samples),
    'temp_min': np.random.normal(18, 3, n_samples),
    'humidity_mean': np.random.normal(75, 10, n_samples),
    'soil_moisture_mean': np.random.normal(55, 15, n_samples),
    'rain_total': np.random.exponential(50, n_samples),
    'days_since_planting': np.random.randint(0, 365, n_samples),
    'area_ha': np.random.uniform(0.5, 20, n_samples),
})

df['yield_kg_ha'] = df['crop_type'].map(base_yields)
df['yield_kg_ha'] += np.random.normal(0, df['yield_kg_ha'] * 0.15, n_samples)
df['yield_kg_ha'] = df['yield_kg_ha'].clip(lower=0)

print(f"Shape: {df.shape}")
df.head(3)

## 2. GDD Calculation

Growing Degree Days (GDD) measure heat accumulation:

$$ GDD = \sum \max\left(\frac{T_{max} + T_{min}}{2} - T_{base}, 0\right) $$

For tropical crops, T_base = 10°C is commonly used.

In [ ]:
T_BASE = 10.0  # base temperature for tropical crops

def calculate_gdd(t_min, t_max, t_base=T_BASE):
    """Calculate daily GDD."""
    t_avg = (t_max + t_min) / 2.0
    return np.maximum(0, t_avg - t_base)

df['daily_gdd'] = calculate_gdd(
    df['temp_min'].values,
    df['temp_max'].values,
)
df['gdd_accumulated'] = df['daily_gdd'] * df['days_since_planting']

print("GDD statistics:")
print(df[['daily_gdd', 'gdd_accumulated']].describe())

## 3. Temperature Stress Indicators

In [ ]:
# Heat stress: days where temp exceeds optimal range
df['heat_stress_days'] = (df['temp_max'] > 35).astype(int) * df['days_since_planting'] / 30

# Cold stress: days where temp drops below optimal
df['cold_stress_days'] = (df['temp_min'] < 15).astype(int) * df['days_since_planting'] / 30

# Diurnal temperature range (important for fruit quality)
df['diurnal_range'] = df['temp_max'] - df['temp_min']

print("Temperature stress features:")
print(df[['heat_stress_days', 'cold_stress_days', 'diurnal_range']].describe())

## 4. Soil Moisture Deficit

In [ ]:
# Field capacity (assumed 60% for loam)
FIELD_CAPACITY = 60.0

# Soil moisture deficit: lower values = more stress
df['moisture_deficit'] = (FIELD_CAPACITY - df['soil_moisture_mean']).clip(lower=0)

# Water stress index (0 = no stress, 1 = severe stress)
df['water_stress_index'] = df['moisture_deficit'] / FIELD_CAPACITY

# Rain × moisture interaction
df['rain_moisture_interaction'] = df['rain_total'] * df['soil_moisture_mean'] / 100

print("Moisture features:")
print(df[['moisture_deficit', 'water_stress_index', 'rain_moisture_interaction']].describe())

## 5. Weather Aggregates (Rolling Means)

In [ ]:
# Simulate a time series for rolling features
df_sorted = df.sort_values('days_since_planting').reset_index(drop=True)

# 7-day rolling averages (using grouped windows)
df_sorted['temp_7d_avg'] = df_sorted['temp_mean'].rolling(7, min_periods=1).mean()
df_sorted['rain_7d_total'] = df_sorted['rain_total'].rolling(7, min_periods=1).sum()
df_sorted['humidity_7d_avg'] = df_sorted['humidity_mean'].rolling(7, min_periods=1).mean()

df = df_sorted
print("Rolling aggregates:")
print(df[['temp_7d_avg', 'rain_7d_total', 'humidity_7d_avg']].describe())

## 6. Feature Importance (Preliminary)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Prepare feature set
feature_cols = [
    'temp_mean', 'temp_max', 'temp_min', 'humidity_mean',
    'soil_moisture_mean', 'rain_total', 'days_since_planting', 'area_ha',
    'daily_gdd', 'gdd_accumulated', 'heat_stress_days', 'cold_stress_days',
    'diurnal_range', 'moisture_deficit', 'water_stress_index',
    'rain_moisture_interaction', 'temp_7d_avg', 'rain_7d_total', 'humidity_7d_avg',
]

df_ml = pd.get_dummies(df, columns=['crop_type'], prefix='crop')
feature_cols += [c for c in df_ml.columns if c.startswith('crop_')]

X = df_ml[feature_cols]
y = df_ml['yield_kg_ha']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Quick RF for feature importance
rf_base = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_base.feature_importances_,
}).sort_values('importance', ascending=False)

print("R² score:", rf_base.score(X_test, y_test))
print("\nTop 10 features:")
feature_importance.head(10)

In [ ]:
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'].values, color='steelblue')
plt.yticks(range(len(top_features)), top_features['feature'].values)
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Engineered Feature Set

**Final feature vector** (to be used in model training):

| Feature | Type | Description |
|---------|------|-------------|
| temp_mean | float | Mean daily temperature |
| temp_max | float | Maximum daily temperature |
| temp_min | float | Minimum daily temperature |
| humidity_mean | float | Mean daily humidity |
| soil_moisture_mean | float | Mean daily soil moisture |
| rain_total | float | Total daily rainfall |
| days_since_planting | int | Days since planting |
| area_ha | float | Field area |
| gdd_accumulated | float | Cumulative GDD |
| heat_stress_days | float | Estimated heat stress days |
| cold_stress_days | float | Estimated cold stress days |
| diurnal_range | float | T_max - T_min |
| water_stress_index | float | 0-1 normalized moisture deficit |
| crop_* (one-hot) | int | Crop type indicator |

**Next**: Proceed to [03 Model Training](03_model_training.ipynb)